# 05 · Tool calling

The model never executes anything. It returns a structured request ("call `get_rate` with `{...}`");
the application runs the function and sends the result back in a `tool` message; the model then
writes the final answer. The protocol by hand first, then the `ToolBox` abstraction.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, for `toolkit`

In [ ]:
import json

from toolkit import ToolBox, completion

PROVIDER = "ollama"  # llama3.2 supports tools; frontier models are more reliable at it

## The protocol by hand

In [ ]:
RATES_TO_NGN = {"USD": 1520.0, "GBP": 1985.0, "EUR": 1690.0, "GHS": 98.0}


def get_rate(currency: str) -> dict:
    rate = RATES_TO_NGN.get(currency.upper())
    return (
        {"currency": currency.upper(), "ngn_per_unit": rate}
        if rate
        else {"error": "unsupported currency"}
    )


rate_tool = {
    "type": "function",
    "function": {
        "name": "get_rate",
        "description": "Exchange rate from a currency to Nigerian naira (NGN).",
        "parameters": {
            "type": "object",
            "properties": {"currency": {"type": "string", "description": "ISO code, e.g. USD"}},
            "required": ["currency"],
        },
    },
}

messages = [{"role": "user", "content": "How many naira is 250 USD? And 80 GBP?"}]
response = completion(messages, provider=PROVIDER, tools=[rate_tool])
choice = response.choices[0]
print(choice.finish_reason)
choice.message.tool_calls

The model asked for two calls in one response (parallel tool calls). Each result goes back as a
`tool` message referencing its `tool_call_id`, after the assistant message that requested it.

In [ ]:
messages.append(choice.message.model_dump(exclude_none=True))
for call in choice.message.tool_calls or []:  # the model may answer without tools
    args = json.loads(call.function.arguments)
    # Read only the expected key: small models occasionally invent extra argument names
    result = get_rate(args.get("currency", ""))
    messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

final = completion(messages, provider=PROVIDER, tools=[rate_tool])
print(final.choices[0].message.content)

## ToolBox: schemas from signatures, loop handled

Hand-written schemas drift from the code. `toolkit.ToolBox` derives them from type hints and docstrings
(`Literal` becomes `enum`, defaults decide `required`), loops until the model stops requesting tools, and returns
exceptions to the model as error results instead of crashing. That matters in practice: llama3.2 sometimes
sends argument names that aren't in the schema, and an error result lets it correct itself on the next round.

In [ ]:
from typing import Literal

box = ToolBox(on_call=lambda c: print(f"  ↳ {c.name}({c.arguments}) -> {c.result}"))


@box.tool
def get_rate_v2(currency: Literal["USD", "GBP", "EUR", "GHS"]) -> dict:
    """Exchange rate from a currency to Nigerian naira (NGN).

    Args:
        currency: ISO currency code.
    """
    return {"currency": currency, "ngn_per_unit": RATES_TO_NGN[currency]}


@box.tool
def convert(amount: float, ngn_per_unit: float) -> float:
    """Multiply an amount by an exchange rate.

    Args:
        amount: Amount in the source currency.
        ngn_per_unit: Rate returned by get_rate_v2.
    """
    return round(amount * ngn_per_unit, 2)


print(json.dumps(box.specs[0], indent=1))

In [ ]:
# Sequential dependency: the model has to get the rate before it can convert
box.run(
    [{"role": "user", "content": "Convert 250 EUR to naira using the tools."}], provider=PROVIDER
)

## Streaming with tools

When streaming, tool calls arrive as fragments (the id and name first, then the JSON arguments in pieces)
keyed by `index`. `ToolBox.stream` stitches them together, executes them, and continues; text is yielded as it arrives.

In [ ]:
for chunk in box.stream(
    [{"role": "user", "content": "What's 1,000 GHS in naira?"}], provider=PROVIDER
):
    print(chunk, end="", flush=True)

Notes:
- Tool descriptions are prompts. Precise descriptions and constrained types (`Literal`) noticeably
  improve argument quality on small models.
- Side-effecting tools (bookings, payments) need a confirmation step in the system prompt and
  validation inside the tool. The model's arguments are untrusted input.
- The full application of this lives in `projects/travel_desk`.